<a href="https://colab.research.google.com/github/anson70242/AI_CUP_2025_Racket/blob/main/Ex02_01HalfPrecision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Low Precision Inference for Deep Learning Models

**Set the Hardware Accelarator to "T4 GPU" before running the program**

The provided program demonstrates the acceleration of a ResNet50 model by converting it from FP32 to FP16. This is a common technique known as low precision inference. It involves reducing the precision of floating-point numbers used in the model to achieve faster computation and lower memory consumption.

##Understanding the Advantages of FP16

FP16 (half-precision floating-point) uses half the number of bits compared to FP32 (single-precision floating-point). This reduction in bit size offers several advantages:
* Reduced Memory Usage: Storing model parameters and activations in FP16 significantly reduces the memory footprint, allowing for larger models or larger batch sizes.
* Faster Computation: Many GPUs are specifically designed to perform calculations on FP16 data much faster than FP32, leading to substantial speedups, especially during inference.

The program efficiently converts the model and input data to FP16 using `model.half()` and `input_image.half()`, respectively.

##Importance of Warm-up and Synchronization

The program incorporates a warm-up run to mitigate the impact of overheads like cache misses and JIT compilation, ensuring more accurate execution time measurements. By performing an initial run, these overheads are accounted for, and subsequent time measurements are more representative of the model's actual performance.

Accurate time measurement on the GPU is crucial. The program utilizes `torch.cuda.synchronize()` to synchronize with the GPU, guaranteeing precise timekeeping for operations performed on the GPU.

##The Trade-off: Speed vs. Accuracy
While low precision inference can significantly enhance inference speed on GPUs, it's essential to acknowledge the potential trade-off with accuracy. Lowering the precision of floating-point numbers can introduce rounding errors that might slightly affect the model's predictions.
However, in many practical scenarios, this reduction in accuracy is often negligible, especially when weighed against the substantial benefits of increased speed and reduced memory consumption. Therefore, low precision inference, specifically utilizing FP16, is a valuable technique worth considering for deep learning applications.




In [ ]:
import torch
import torchvision
from torchvision.models import resnet50, ResNet50_Weights
import time

# Preparation of ResNet50 model
model = torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)
model.to('cuda') # send the model to GPU
model.eval()

# Preparation of input data
input_data  = torch.randn((512, 3, 224, 224))
input_image = input_data.to('cuda')

# Worm-up run
torch.cuda.synchronize()
start_worm = time.time()
with torch.no_grad():
    output = model(input_image)
torch.cuda.synchronize()
end_worm = time.time()

# Run (on FP32)
torch.cuda.synchronize()
start_fp32 = time.time()
with torch.no_grad():
    output = model(input_image)
torch.cuda.synchronize()
end_fp32 = time.time()

wormup_time = end_worm-start_worm
exec_fp32_time = end_fp32 - start_fp32

print(f"              Worm-up time: {wormup_time}")
print(f"Execution time on FP32,GPU: {exec_fp32_time}")


          Worm-up time: 2.254746437072754
Execution time on FP32: 1.2579731941223145


In [ ]:
# Translation to FP16
model_fp16 = model.half()
input_image_fp16 = input_image.half()

# Worm-up Run
with torch.no_grad():
  output=model_fp16(input_image_fp16)


# Execution of FP16
torch.cuda.synchronize()
start_fp16 = time.time()
with torch.no_grad():
    output = model_fp16(input_image_fp16)
torch.cuda.synchronize()
end_fp16 = time.time()

exec_fp16_time = end_fp16 - start_fp16

print(f"Execution time on FP32,GPU: {exec_fp32_time}")
print(f"Execution time on FP16,GPU: {exec_fp16_time}")

Execution time on FP32: 1.2579731941223145
Execution time on FP16: 0.5775613784790039



Assignment 1: Relationship between batch size and execution time

**What to Submit:**
* A graph showing the relationship between batch size and execution time per image.
  * The horizontal axis represents batch size, the vertical axis represents execution time per image, and the results of FP32 and FP16 are displayed in different colors.

* Discussion: Discuss the relationship between batch size and execution time per image from the graph from the following perspectives:
  * **GPU Parallel Processing Capability:** GPUs have many CUDA cores and can execute FP16 operations at high speed. The larger the batch size, the more this parallel processing capability is utilized, and the speed advantage of FP16 becomes more pronounced.
  * **Data Transfer and Cache:** When the batch size is small, overheads such as data transfer and cache misses become relatively large, and the GPU's computational power may not be fully utilized. Increasing the batch size can reduce these overheads.

**Experiment:**
1. Load the ResNet50 model in both FP32 and FP16.
2. Measure the execution time of both FP32 and FP16 ResNet50 models at various batch sizes (e.g., 1, 4, 8, 16, 32, 64, 128, 256, 512).

Input images are created using

```
input_data = torch.randn((512, 3, 224, 224))
```

The elements of the tuple represent batch size 512, channel size 3, height 224, and width 224, respectively. The execution time per image is calculated as execution time / batch size.

Analyze these experimental results along with the graph showing the relationship between batch size and execution time per image, and develop specific considerations on the Tesla T4's Tensor Core performance and FP16 memory efficiency.

**Points for Discussion:**
* Since the Tesla T4's Tensor Cores can process FP16 operations at high speed, it is expected that the execution speed of FP16 will be significantly faster than FP32 as the batch size increases.
* Because FP16 requires less memory than FP32, it may be possible to use larger batch sizes, which may be advantageous in the shared GPU environment of Google Colaboratory.

When analyzing the experimental results, consider these points and develop specific discussions on the performance of the Tesla T4's Tensor Cores and the memory efficiency of FP16.


